# 08 · Filters change the search problem

## The engineering question
Does filtering retrieve the best eligible points, or merely remove ineligible points from an already limited result list?

**Learning outcomes:** express tenant/current-version policy as a conjunction; measure filter selectivity; compare pre-filtered search with post-filtering; build payload indexes before ingestion; verify results against the same eligible exact oracle.

This synthetic experiment separates correctness from semantic relevance. Each point has a tenant, current-version flag, and bucket. Request identity must be trusted by the server; accepting a caller-supplied tenant is not authentication. Lesson 24 implements that boundary. Filtering inside Qdrant changes the candidate population; filtering only the returned top-k can starve results.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vector_masterclass.lesson import LessonRun
from vector_masterclass.config import root

plt.rcParams.update(
    {"figure.figsize": (8, 4), "axes.spines.top": False, "axes.spines.right": False}
)
lesson = LessonRun("08", "B", docker=True)
print("Kernel:", sys.executable)
print("Workspace:", root())

In [ ]:
from uuid import uuid4
import time
from qdrant_client import QdrantClient, models as qm
from vector_masterclass.config import qdrant_url
from vector_masterclass.metrics import ann_recall

rng = np.random.default_rng(8)
vectors = rng.normal(size=(2000, 32)).astype("float32")
vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
query = rng.normal(size=32).astype("float32")
query /= np.linalg.norm(query)
payloads = [
    {"tenant": "red" if i % 2 else "blue", "active": i % 7 != 0, "bucket": i % 100}
    for i in range(len(vectors))
]
client = QdrantClient(url=qdrant_url(), timeout=20)
name = "vmc_08_" + uuid4().hex[:8]
client.create_collection(name, vectors_config=qm.VectorParams(size=32, distance=qm.Distance.COSINE))
rows = []
try:
    for field, kind in [
        ("tenant", qm.PayloadSchemaType.KEYWORD),
        ("active", qm.PayloadSchemaType.BOOL),
        ("bucket", qm.PayloadSchemaType.INTEGER),
    ]:
        client.create_payload_index(name, field, kind, wait=True)
    for start in range(0, len(vectors), 500):
        client.upsert(
            name,
            points=[
                qm.PointStruct(id=i, vector=vectors[i].tolist(), payload=payloads[i])
                for i in range(start, start + 500)
            ],
            wait=True,
        )
    unfiltered = client.query_points(
        name, query=query.tolist(), limit=10, search_params=qm.SearchParams(exact=True)
    ).points
    for upper in [100, 20, 4]:
        eligible = [
            i
            for i, p in enumerate(payloads)
            if p["tenant"] == "red" and p["active"] and p["bucket"] < upper
        ]
        oracle = sorted(eligible, key=lambda i: -float(vectors[i] @ query))[:10]
        policy = qm.Filter(
            must=[
                qm.FieldCondition(key="tenant", match=qm.MatchValue(value="red")),
                qm.FieldCondition(key="active", match=qm.MatchValue(value=True)),
                qm.FieldCondition(key="bucket", range=qm.Range(lt=upper)),
            ]
        )
        latencies = []
        for repeat in range(5):
            start = time.perf_counter()
            hits = client.query_points(
                name,
                query=query.tolist(),
                query_filter=policy,
                limit=10,
                search_params=qm.SearchParams(exact=True),
            ).points
            latencies.append((time.perf_counter() - start) * 1000)
        post = [p.id for p in unfiltered if p.id in set(eligible)]
        rows.append(
            {
                "bucket_limit": upper,
                "eligible_points": len(eligible),
                "selectivity": len(eligible) / len(vectors),
                "pre_filter_count": len(hits),
                "post_filter_count": len(post),
                "pre_filter_oracle_recall": ann_recall([h.id for h in hits], oracle, 10),
                "post_filter_oracle_recall": ann_recall(post, oracle, 10),
                "median_ms": float(np.median(latencies)),
                "policy_valid": all(h.id in eligible for h in hits),
            }
        )
finally:
    client.delete_collection(name)
    client.close()
results = lesson.table(pd.DataFrame(rows))
display(results)
fig, ax = plt.subplots()
results.set_index("eligible_points")[["pre_filter_count", "post_filter_count"]].rename(
    columns={"pre_filter_count": "Filter in query", "post_filter_count": "Filter returned top 10"}
).plot.bar(ax=ax)
ax.set(
    ylabel="Returned eligible points",
    xlabel="Eligible points in the corpus",
    title="Post-filtering a limited ranking can starve results",
)
ax.tick_params(axis="x", rotation=0)
ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
lesson.figure(fig)
plt.show()

## Interpretation and exercise
Eligibility correctness is a separate property from nearest-neighbor fidelity and relevance. A result can be semantically useful yet forbidden. Small eligible sets may make an exact scan cheaper than graph traversal; lesson 09 examines why selectivity complicates ANN claims. Five warm requests are illustrative timings, not a capacity estimate.

**Exercise:** change the policy conjunction to an OR on paper. Give a concrete cross-tenant point it would admit. Then test a nonexistent tenant and confirm an empty result rather than a fallback to unrestricted search. Explain why payload-index design belongs in the schema review.

[Qdrant filtering reference](https://qdrant.tech/documentation/search/filtering/). Production policy must also cover sparse prefetches, caches, reranking inputs, and fallback queries; applying it only to the final displayed list is insufficient.

In [ ]:
manifest = lesson.finish(
    {
        "eligible_oracle_matches": bool(results.pre_filter_oracle_recall.eq(1).all()),
        "tenant_and_version_hold": bool(results.policy_valid.all()),
        "selectivity_varies": results.eligible_points.nunique() == 3,
        "post_filter_failure_observed": bool(
            (results.post_filter_count < results.pre_filter_count).any()
        ),
    }
)

## References and next steps
Consult the [Qdrant documentation](https://qdrant.tech/documentation/),
the [MiniLM model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2),
and [uv's Jupyter guide](https://docs.astral.sh/uv/guides/integration/jupyter/).
The exact pinned configuration is in `configs/models.json`; worked exercises are in `docs/SOLUTIONS.md`.
This notebook's measurements apply to the recorded local run, not a production deployment.